# Pegelonline Feeder (Fabric Notebook)

Polls the German WSV PegelOnline API for water-level measurements and emits CloudEvents to the bound Fabric Event Stream.

**Operating model**
- Runs `pegelonline feed --once` on each scheduled execution.
- Polling cadence (15 min) is enforced by the Data Factory pipeline schedule, not by the bridge.
- Dedupe state is persisted to a Lakehouse file (`STATE_FILE`) so reruns only emit new measurements.

**Configuration**
- `CONNECTION_STRING`: Fabric Event Stream custom-endpoint connection string. Populated by `deploy-feeder-notebook.ps1` at deploy time.
- `STATE_FILE`: Lakehouse-mounted path for the dedupe-state JSON. Defaults to the workspace default Lakehouse.
- `POLLING_INTERVAL`: Used only if `ONCE_MODE=false`. Default 900s.
- `ONCE_MODE`: When true (default in scheduled use), the bridge exits after one cycle.

**Prerequisites**
- KQL database created and schema applied (`deploy-fabric.ps1` style step).
- Event Stream created with the bound topology; connection string available.

In [ ]:
# === PARAMETERS ===
# CONNECTION_STRING is rewritten by the deploy script with the live Event Stream CS.
CONNECTION_STRING = ""
STATE_FILE        = "/lakehouse/default/Files/feeder-state/pegelonline/state.json"
POLLING_INTERVAL  = 900
ONCE_MODE         = True
# Workspace-specific defaults - overwritten by the deploy script.
KUSTO_URI         = ""
KUSTO_DATABASE    = ""
# Pin the source revision; deploy script overwrites with the merged SHA.
SOURCE_REF        = "main"

In [ ]:
# Install the pegelonline bridge directly from this repo's subdirectory.
# Poetry-style path deps for pegelonline_producer_data and
# pegelonline_producer_kafka_producer resolve against the cloned tree.
%pip install -q --no-cache-dir git+https://github.com/clemensv/real-time-sources@{SOURCE_REF}#subdirectory=pegelonline

In [ ]:
# Prepare Lakehouse-backed state directory.
import os, pathlib
state_path = pathlib.Path(STATE_FILE)
state_path.parent.mkdir(parents=True, exist_ok=True)
print(f"State file: {STATE_FILE} (exists={state_path.exists()})")
# Wire env vars - bridge picks these up via os.getenv.
os.environ["CONNECTION_STRING"] = CONNECTION_STRING
os.environ["STATE_FILE"]        = STATE_FILE
os.environ["POLLING_INTERVAL"]  = str(POLLING_INTERVAL)
os.environ["ONCE_MODE"]         = "true" if ONCE_MODE else "false"
assert CONNECTION_STRING, "CONNECTION_STRING is empty. Redeploy via deploy-feeder-notebook.ps1."

In [ ]:
# Run one poll cycle via the bridge's main() entry point.
# Equivalent to: python -m pegelonline feed --once
import sys
from pegelonline import pegelonline as feeder
argv_backup = sys.argv
try:
    sys.argv = ["pegelonline", "feed", "--once"]
    feeder.main()
finally:
    sys.argv = argv_backup
print("Cycle complete.")